In [1]:
import os
import time
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import xarray as xr
import earthaccess

from pyproj import Transformer
from tqdm.auto import tqdm

In [2]:
OUTPUT_DIR = Path("Data/Global/smap")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SMAP_DATASET_DIR = Path(
    "Data/Global/pipeline_stage1/smap_parquet_dataset"
)
SMAP_DATASET_DIR.mkdir(parents=True, exist_ok=True)

SMAP_SHORT_NAME = "SPL4SMGP"
SMAP_VERSION = "008"

SMAP_VARIABLE = "sm_surface"

SM_MIN = 0.0
SM_MAX = 0.9

SMAP_CRS = "EPSG:6933"

# NASA's current V8 reprocessing interval.
SMAP_REPROCESS_START = pd.Timestamp("2026-05-14")
SMAP_REPROCESS_END = pd.Timestamp("2026-07-28")

print("SMAP dataset directory:", SMAP_DATASET_DIR)

SMAP dataset directory: Data/Global/pipeline_stage1/smap_parquet_dataset


In [3]:
if "samples" not in globals():
    samples = pd.read_parquet(
        "Data/Global/pipeline_stage1/station_date_candidates.parquet"
    )

if "stations" not in globals():
    stations = pd.read_parquet(
        "Data/Global/pipeline_stage1/stations.parquet"
    )

samples["sample_date"] = pd.to_datetime(
    samples["sample_date"]
).dt.normalize()

print("Samples:", f"{len(samples):,}")
print("Stations:", f"{len(stations):,}")

Samples: 3,514,643
Stations: 2,210


In [4]:
if "station_key" not in samples.columns:
    samples["network"] = samples["network"].astype(str)
    samples["station_id"] = samples["station_id"].astype(str)

    samples["station_key"] = (
        samples["network"]
        + "::"
        + samples["station_id"]
    )

if "station_key" not in stations.columns:
    stations["network"] = stations["network"].astype(str)
    stations["station_id"] = stations["station_id"].astype(str)

    stations["station_key"] = (
        stations["network"]
        + "::"
        + stations["station_id"]
    )

print(
    "Unique station keys:",
    samples["station_key"].nunique()
)

Unique station keys: 2210


In [5]:
auth = earthaccess.login()

print("Earthdata authenticated:", auth.authenticated)

Earthdata authenticated: True


In [6]:
print(
    earthaccess.status()
)

{'Earthdata Login': 'OK', 'Common Metadata Repository': 'OK'}


In [7]:
TEST_DATE = pd.Timestamp("2020-01-15").normalize()
TEST_END = TEST_DATE + pd.Timedelta(days=1)

test_granules = earthaccess.search_data(
    short_name=SMAP_SHORT_NAME,
    version=SMAP_VERSION,
    temporal=(
        TEST_DATE.strftime("%Y-%m-%dT00:00:00"),
        TEST_END.strftime("%Y-%m-%dT00:00:00"),
    ),
    count=-1,
)

print("Test date:", TEST_DATE.date())
print("Granules found:", len(test_granules))

for i, g in enumerate(test_granules):
    print(i, g)

Test date: 2020-01-15
Granules found: 10
0 Collection: {'Version': '008', 'ShortName': 'SPL4SMGP'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'BoundingRectangles': [{'WestBoundingCoordinate': -180, 'SouthBoundingCoordinate': -85.04456, 'EastBoundingCoordinate': 180, 'NorthBoundingCoordinate': 85.04456}]}}}
Temporal coverage: {'RangeDateTime': {'EndingDateTime': '2020-01-15T00:00:00.000Z', 'BeginningDateTime': '2020-01-14T21:00:00.000Z'}}
Size(MB): 139.40195560455322
Data: ['https://data.nsidc.earthdatacloud.nasa.gov/nsidc-cumulus-prod-protected/SMAP/SPL4SMGP/008/2020/01/14/SMAP_L4_SM_gph_20200114T223000_Vv8010_001.h5']
1 Collection: {'Version': '008', 'ShortName': 'SPL4SMGP'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'BoundingRectangles': [{'WestBoundingCoordinate': -180, 'SouthBoundingCoordinate': -85.04456, 'EastBoundingCoordinate': 180, 'NorthBoundingCoordinate': 85.04456}]}}}
Temporal coverage: {'RangeDateTime': {'EndingDateTime': '2020-01-15T03:0

/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/earthaccess/results.py:364: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  Size(MB): {self.size()}


In [8]:
remote_files = earthaccess.open(
    test_granules[:1],
    show_progress=False,
)

if not remote_files:
    raise RuntimeError(
        "earthaccess.open() returned no files."
    )

f = remote_files[0]

print("Opened test granule:")
print(f)

/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/earthaccess/store.py:516: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum([granule.size() for granule in granules]) / 1024, 2)


Opened test granule:
<File-like object HTTPFileSystem, https://data.nsidc.earthdatacloud.nasa.gov/nsidc-cumulus-prod-protected/SMAP/SPL4SMGP/008/2020/01/14/SMAP_L4_SM_gph_20200114T223000_Vv8010_001.h5>


In [9]:
geo_ds = xr.open_dataset(
    f,
    engine="h5netcdf",
    group="Geophysical_Data",
    phony_dims="access",
    cache=False,
)

print(geo_ds)

<xarray.Dataset> Size: 1GB
Dimensions:                            (y: 1624, x: 3856)
Dimensions without coordinates: y, x
Data variables: (12/45)
    baseflow_flux                      (y, x) float32 25MB ...
    depth_to_water_table_from_surface  (y, x) float32 25MB ...
    free_surface_water_on_peat_flux    (y, x) float32 25MB ...
    heat_flux_ground                   (y, x) float32 25MB ...
    heat_flux_latent                   (y, x) float32 25MB ...
    heat_flux_sensible                 (y, x) float32 25MB ...
    ...                                 ...
    specific_humidity_lowatmmodlay     (y, x) float32 25MB ...
    surface_pressure                   (y, x) float32 25MB ...
    surface_temp                       (y, x) float32 25MB ...
    temp_lowatmmodlay                  (y, x) float32 25MB ...
    vegetation_greenness_fraction      (y, x) float32 25MB ...
    windspeed_lowatmmodlay             (y, x) float32 25MB ...


In [10]:
sm = geo_ds[SMAP_VARIABLE]

x = geo_ds["x"].values
y = geo_ds["y"].values

SMAP_Y_DIM = sm.dims[-2]
SMAP_X_DIM = sm.dims[-1]

print("SMAP variable:")
print(sm)

print("\nSMAP dimensions:", sm.dims)
print("SMAP shape:", sm.shape)

print("\nX:")
print(
    "shape =", x.shape,
    "min =", np.nanmin(x),
    "max =", np.nanmax(x),
)

print("\nY:")
print(
    "shape =", y.shape,
    "min =", np.nanmin(y),
    "max =", np.nanmax(y),
)

print("\nAttributes:")
print(sm.attrs)

print("\nSelected dimensions:")
print("Y:", SMAP_Y_DIM)
print("X:", SMAP_X_DIM)

SMAP variable:
<xarray.DataArray 'sm_surface' (y: 1624, x: 3856)> Size: 25MB
[6262144 values with dtype=float32]
Dimensions without coordinates: y, x
Attributes:
    DIMENSION_LABELS:  [b'y', b'x']
    fmissing_value:    -9999.0
    grid_mapping:      EASE2_global_projection
    long_name:         Top layer soil moisture (0-5 cm)
    units:             m3 m-3
    valid_max:         0.9
    valid_min:         0.0

SMAP dimensions: ('y', 'x')
SMAP shape: (1624, 3856)

X:
shape = (3856,) min = 0 max = 3855

Y:
shape = (1624,) min = 0 max = 1623

Attributes:
{'DIMENSION_LABELS': [b'y', b'x'], 'fmissing_value': np.float32(-9999.0), 'grid_mapping': 'EASE2_global_projection', 'long_name': 'Top layer soil moisture (0-5 cm)', 'units': 'm3 m-3', 'valid_max': np.float32(0.9), 'valid_min': np.float32(0.0)}

Selected dimensions:
Y: y
X: x


In [11]:
# ============================================================
# BUILD SMAP STATION -> GRID INDICES
# ============================================================
#
# IMPORTANT:
# SPL4SMGP V008 stores x/y as pixel indices:
#
#   x = 0 ... 3855
#   y = 0 ... 1623
#
# They are NOT projected coordinates in metres.
#
# We therefore:
#
#   station lon/lat
#       -> EPSG:6933 metres
#       -> SMAP grid cell index
#
# SMAP SPL4SMGP V008 grid:
#   columns = 3856
#   rows    = 1624
#   cell size ~= 9008.05 m
# ============================================================

SMAP_NCOLS = len(x)
SMAP_NROWS = len(y)

SMAP_CELL_SIZE_M = 9008.05

# Global EASE-Grid 2.0 upper-left outer edge.
SMAP_ULX = -17367530.44
SMAP_ULY = 7314540.83

# ------------------------------------------------------------
# Convert station longitude/latitude to EPSG:6933 metres
# ------------------------------------------------------------

transformer = Transformer.from_crs(
    "EPSG:4326",
    SMAP_CRS,
    always_xy=True,
)

station_x_m, station_y_m = transformer.transform(
    stations["longitude"].to_numpy(dtype=float),
    stations["latitude"].to_numpy(dtype=float),
)

stations_smap = stations.copy()

stations_smap["smap_x_m"] = station_x_m
stations_smap["smap_y_m"] = station_y_m

# ------------------------------------------------------------
# Convert projected coordinates to SMAP pixel indices
# ------------------------------------------------------------

stations_smap["smap_x_idx"] = np.floor(
    (
        stations_smap["smap_x_m"]
        - SMAP_ULX
    )
    / SMAP_CELL_SIZE_M
).astype(np.int32)

stations_smap["smap_y_idx"] = np.floor(
    (
        SMAP_ULY
        - stations_smap["smap_y_m"]
    )
    / SMAP_CELL_SIZE_M
).astype(np.int32)

# ------------------------------------------------------------
# Validate grid indices
# ------------------------------------------------------------

x_idx_valid = (
    (stations_smap["smap_x_idx"] >= 0)
    &
    (stations_smap["smap_x_idx"] < SMAP_NCOLS)
)

y_idx_valid = (
    (stations_smap["smap_y_idx"] >= 0)
    &
    (stations_smap["smap_y_idx"] < SMAP_NROWS)
)

print("Stations:", len(stations_smap))

print(
    "X index out of bounds:",
    (~x_idx_valid).sum(),
)

print(
    "Y index out of bounds:",
    (~y_idx_valid).sum(),
)

if not x_idx_valid.all() or not y_idx_valid.all():

    bad = stations_smap.loc[
        ~(x_idx_valid & y_idx_valid),
        [
            "station_key",
            "latitude",
            "longitude",
            "smap_x_m",
            "smap_y_m",
            "smap_x_idx",
            "smap_y_idx",
        ],
    ]

    display(bad.head(20))

    raise ValueError(
        "Some stations fall outside the SMAP grid."
    )

print("All stations have valid SMAP grid indices.")

print(
    "\nX index range:",
    stations_smap["smap_x_idx"].min(),
    "to",
    stations_smap["smap_x_idx"].max(),
)

print(
    "Y index range:",
    stations_smap["smap_y_idx"].min(),
    "to",
    stations_smap["smap_y_idx"].max(),
)

display(
    stations_smap[
        [
            "station_key",
            "latitude",
            "longitude",
            "smap_x_m",
            "smap_y_m",
            "smap_x_idx",
            "smap_y_idx",
        ]
    ].head(10)
)

Stations: 2210
X index out of bounds: 0
Y index out of bounds: 0
All stations have valid SMAP grid indices.

X index range: 159 to 3812
Y index range: 40 to 1389


,station_key,latitude,longitude,smap_x_m,smap_y_m,smap_x_idx,smap_y_idx
0,AMMA-CATCH::AMMA-CATCH__Banizoumbou,13.53250,2.66040,2.566921e+05,1.710798e+06,1956,622
1,AMMA-CATCH::AMMA-CATCH__Belefoungou-Mid,9.79506,1.70994,1.649858e+05,1.243671e+06,1946,673
2,AMMA-CATCH::AMMA-CATCH__Belefoungou-Top,9.78986,1.70994,1.649858e+05,1.243017e+06,1946,674
3,AMMA-CATCH::AMMA-CATCH__Nalohou-Mid,9.74530,1.60530,1.548894e+05,1.237412e+06,1945,674
4,AMMA-CATCH::AMMA-CATCH__Nalohou-Top,9.74407,1.60580,1.549377e+05,1.237258e+06,1945,674
5,AMMA-CATCH::AMMA-CATCH__Tondikiboro,13.54800,2.69560,2.600884e+05,1.712721e+06,1956,621
6,AMMA-CATCH::AMMA-CATCH__Wankama,13.64560,2.63200,2.539519e+05,1.724833e+06,1956,620
7,ARM::ARM__Anthony,37.21340,-98.09690,-9.465005e+06,4.427857e+06,877,320
8,ARM::ARM__Ashton,37.13300,-97.26600,-9.384835e+06,4.419644e+06,886,321
9,ARM::ARM__Byron,36.88100,-98.28500,-9.483154e+06,4.393847e+06,875,324


In [12]:
# ============================================================
# SMAP GRID GEOMETRY DIAGNOSTIC
# ============================================================

print("SMAP variable dimensions:")
print(sm.dims)
print("SMAP shape:")
print(sm.shape)

print("\nX coordinate:")
print("type:", type(x))
print("shape:", np.asarray(x).shape)
print("min:", np.nanmin(x))
print("max:", np.nanmax(x))

print("\nY coordinate:")
print("type:", type(y))
print("shape:", np.asarray(y).shape)
print("min:", np.nanmin(y))
print("max:", np.nanmax(y))

print("\nFirst X values:")
print(np.asarray(x).reshape(-1)[:10])

print("\nLast X values:")
print(np.asarray(x).reshape(-1)[-10:])

print("\nFirst Y values:")
print(np.asarray(y).reshape(-1)[:10])

print("\nLast Y values:")
print(np.asarray(y).reshape(-1)[-10:])

SMAP variable dimensions:
('y', 'x')
SMAP shape:
(1624, 3856)

X coordinate:
type: <class 'numpy.ndarray'>
shape: (3856,)
min: 0
max: 3855

Y coordinate:
type: <class 'numpy.ndarray'>
shape: (1624,)
min: 0
max: 1623

First X values:
[0 1 2 3 4 5 6 7 8 9]

Last X values:
[3846 3847 3848 3849 3850 3851 3852 3853 3854 3855]

First Y values:
[0 1 2 3 4 5 6 7 8 9]

Last Y values:
[1614 1615 1616 1617 1618 1619 1620 1621 1622 1623]


In [13]:
"""# Coordinates are in the Metadata group for SPL4SMGP V8
meta_ds = xr.open_dataset(
    f,
    engine="h5netcdf",
    group="Metadata",
    phony_dims="access",
)

x = meta_ds["x"].values
y = meta_ds["y"].values

print("x:", x.shape, x.min(), x.max())
print("y:", y.shape, y.min(), y.max())"""

'# Coordinates are in the Metadata group for SPL4SMGP V8\nmeta_ds = xr.open_dataset(\n    f,\n    engine="h5netcdf",\n    group="Metadata",\n    phony_dims="access",\n)\n\nx = meta_ds["x"].values\ny = meta_ds["y"].values\n\nprint("x:", x.shape, x.min(), x.max())\nprint("y:", y.shape, y.min(), y.max())'

In [14]:
"""transformer = Transformer.from_crs(
    "EPSG:4326",
    SMAP_CRS,
    always_xy=True,
)

station_x, station_y = transformer.transform(
    stations["longitude"].to_numpy(dtype=float),
    stations["latitude"].to_numpy(dtype=float),
)

stations_smap = stations.copy()

stations_smap["smap_x"] = station_x
stations_smap["smap_y"] = station_y"""

'transformer = Transformer.from_crs(\n    "EPSG:4326",\n    SMAP_CRS,\n    always_xy=True,\n)\n\nstation_x, station_y = transformer.transform(\n    stations["longitude"].to_numpy(dtype=float),\n    stations["latitude"].to_numpy(dtype=float),\n)\n\nstations_smap = stations.copy()\n\nstations_smap["smap_x"] = station_x\nstations_smap["smap_y"] = station_y'

In [15]:
"""def nearest_index(values, targets):
    values = np.asarray(values)
    targets = np.asarray(targets)

    ascending = values[0] < values[-1]

    if ascending:
        work = values
    else:
        work = values[::-1]

    idx = np.searchsorted(
        work,
        targets,
        side="left",
    )

    idx = np.clip(
        idx,
        0,
        len(work) - 1,
    )

    left = np.maximum(idx - 1, 0)
    right = idx

    choose_left = (
        np.abs(work[left] - targets)
        <
        np.abs(work[right] - targets)
    )

    idx = np.where(
        choose_left,
        left,
        right,
    )

    if not ascending:
        idx = len(values) - 1 - idx

    return idx.astype(np.int32)"""

'def nearest_index(values, targets):\n    values = np.asarray(values)\n    targets = np.asarray(targets)\n\n    ascending = values[0] < values[-1]\n\n    if ascending:\n        work = values\n    else:\n        work = values[::-1]\n\n    idx = np.searchsorted(\n        work,\n        targets,\n        side="left",\n    )\n\n    idx = np.clip(\n        idx,\n        0,\n        len(work) - 1,\n    )\n\n    left = np.maximum(idx - 1, 0)\n    right = idx\n\n    choose_left = (\n        np.abs(work[left] - targets)\n        <\n        np.abs(work[right] - targets)\n    )\n\n    idx = np.where(\n        choose_left,\n        left,\n        right,\n    )\n\n    if not ascending:\n        idx = len(values) - 1 - idx\n\n    return idx.astype(np.int32)'

In [16]:
# ------------------------------------------------------------
# Build station coordinates in SMAP EPSG:6933
# ------------------------------------------------------------

"""
transformer = Transformer.from_crs(
    "EPSG:4326",
    SMAP_CRS,
    always_xy=True,
)

station_x, station_y = transformer.transform(
    stations["longitude"].to_numpy(dtype=float),
    stations["latitude"].to_numpy(dtype=float),
)

stations_smap = stations.copy()

stations_smap["smap_x"] = station_x
stations_smap["smap_y"] = station_y

print("stations_smap created:")
print("Rows:", len(stations_smap))

display(
    stations_smap[
        [
            "station_key",
            "latitude",
            "longitude",
            "smap_x",
            "smap_y",
        ]
    ].head()
)"""

'\ntransformer = Transformer.from_crs(\n    "EPSG:4326",\n    SMAP_CRS,\n    always_xy=True,\n)\n\nstation_x, station_y = transformer.transform(\n    stations["longitude"].to_numpy(dtype=float),\n    stations["latitude"].to_numpy(dtype=float),\n)\n\nstations_smap = stations.copy()\n\nstations_smap["smap_x"] = station_x\nstations_smap["smap_y"] = station_y\n\nprint("stations_smap created:")\nprint("Rows:", len(stations_smap))\n\ndisplay(\n    stations_smap[\n        [\n            "station_key",\n            "latitude",\n            "longitude",\n            "smap_x",\n            "smap_y",\n        ]\n    ].head()\n)'

In [17]:
# ============================================================
# SMAP GRID INDEX QA
# ============================================================

# The SMAP x/y variables are pixel indices, NOT metres.
# Therefore we do NOT compare smap_x/smap_y against x/y.

x_idx_valid = (
    (stations_smap["smap_x_idx"] >= 0)
    & (stations_smap["smap_x_idx"] < SMAP_NCOLS)
)

y_idx_valid = (
    (stations_smap["smap_y_idx"] >= 0)
    & (stations_smap["smap_y_idx"] < SMAP_NROWS)
)

print("X index out of bounds:", (~x_idx_valid).sum())
print("Y index out of bounds:", (~y_idx_valid).sum())

if not x_idx_valid.all() or not y_idx_valid.all():
    bad = stations_smap.loc[
        ~(x_idx_valid & y_idx_valid),
        [
            "station_key",
            "latitude",
            "longitude",
            "smap_x_m",
            "smap_y_m",
            "smap_x_idx",
            "smap_y_idx",
        ],
    ]

    display(bad.head(20))

    raise ValueError(
        "Some stations fall outside the SMAP grid."
    )

print("All stations have valid SMAP grid indices.")

print("\nIndex ranges:")
print(
    "X:",
    stations_smap["smap_x_idx"].min(),
    "to",
    stations_smap["smap_x_idx"].max(),
)
print(
    "Y:",
    stations_smap["smap_y_idx"].min(),
    "to",
    stations_smap["smap_y_idx"].max(),
)

X index out of bounds: 0
Y index out of bounds: 0
All stations have valid SMAP grid indices.

Index ranges:
X: 159 to 3812
Y: 40 to 1389


In [18]:
"""check = stations_smap.iloc[:10].copy()

check["grid_x"] = x[
    check["smap_x_idx"].to_numpy()
]

check["grid_y"] = y[
    check["smap_y_idx"].to_numpy()
]

check["dx"] = check["grid_x"] - check["smap_x"]
check["dy"] = check["grid_y"] - check["smap_y"]

display(
    check[
        [
            "station_key",
            "latitude",
            "longitude",
            "smap_x",
            "smap_y",
            "smap_x_idx",
            "smap_y_idx",
            "dx",
            "dy",
        ]
    ]
)"""

'check = stations_smap.iloc[:10].copy()\n\ncheck["grid_x"] = x[\n    check["smap_x_idx"].to_numpy()\n]\n\ncheck["grid_y"] = y[\n    check["smap_y_idx"].to_numpy()\n]\n\ncheck["dx"] = check["grid_x"] - check["smap_x"]\ncheck["dy"] = check["grid_y"] - check["smap_y"]\n\ndisplay(\n    check[\n        [\n            "station_key",\n            "latitude",\n            "longitude",\n            "smap_x",\n            "smap_y",\n            "smap_x_idx",\n            "smap_y_idx",\n            "dx",\n            "dy",\n        ]\n    ]\n)'

In [19]:
"""with xr.open_dataset(
    f,
    engine="h5netcdf",
    group="Geophysical_Data",
    phony_dims="access",
) as geo:

    sm = geo["sm_surface"]

    print("dims:", sm.dims)
    print("shape:", sm.shape)
    print("attrs:", sm.attrs)
    print("encoding:", sm.encoding)

    raw = sm.values

    finite = np.isfinite(raw)

    print("finite:", finite.sum())
    print("total:", raw.size)

    if finite.any():
        print("min:", np.nanmin(raw))
        print("max:", np.nanmax(raw))
        print("mean:", np.nanmean(raw))"""

'with xr.open_dataset(\n    f,\n    engine="h5netcdf",\n    group="Geophysical_Data",\n    phony_dims="access",\n) as geo:\n\n    sm = geo["sm_surface"]\n\n    print("dims:", sm.dims)\n    print("shape:", sm.shape)\n    print("attrs:", sm.attrs)\n    print("encoding:", sm.encoding)\n\n    raw = sm.values\n\n    finite = np.isfinite(raw)\n\n    print("finite:", finite.sum())\n    print("total:", raw.size)\n\n    if finite.any():\n        print("min:", np.nanmin(raw))\n        print("max:", np.nanmax(raw))\n        print("mean:", np.nanmean(raw))'

In [20]:
"""stations_smap[
    [
        "station_key",
        "latitude",
        "longitude",
        "smap_x",
        "smap_y",
        "smap_x_idx",
        "smap_y_idx",
    ]
].head(20)"""

'stations_smap[\n    [\n        "station_key",\n        "latitude",\n        "longitude",\n        "smap_x",\n        "smap_y",\n        "smap_x_idx",\n        "smap_y_idx",\n    ]\n].head(20)'

In [21]:
# ============================================================
# SMAP STATION -> GRID CELL QA
# ============================================================

stations_smap["smap_cell_x_m"] = (
    SMAP_ULX
    + (
        stations_smap["smap_x_idx"]
        + 0.5
    ) * SMAP_CELL_SIZE_M
)

stations_smap["smap_cell_y_m"] = (
    SMAP_ULY
    - (
        stations_smap["smap_y_idx"]
        + 0.5
    ) * SMAP_CELL_SIZE_M
)

stations_smap["smap_distance_m"] = np.sqrt(
    (
        stations_smap["smap_cell_x_m"]
        - stations_smap["smap_x_m"]
    ) ** 2
    +
    (
        stations_smap["smap_cell_y_m"]
        - stations_smap["smap_y_m"]
    ) ** 2
)

print("Station-to-SMAP-cell distance:")
display(
    stations_smap[
        [
            "station_key",
            "latitude",
            "longitude",
            "smap_x_idx",
            "smap_y_idx",
            "smap_distance_m",
        ]
    ].head(10)
)

display(
    stations_smap["smap_distance_m"].describe()
)

Station-to-SMAP-cell distance:


,station_key,latitude,longitude,smap_x_idx,smap_y_idx,smap_distance_m
0,AMMA-CATCH::AMMA-CATCH__Banizoumbou,13.53250,2.66040,1956,622,3767.902492
1,AMMA-CATCH::AMMA-CATCH__Belefoungou-Mid,9.79506,1.70994,1946,673,4280.447283
2,AMMA-CATCH::AMMA-CATCH__Belefoungou-Top,9.78986,1.70994,1946,674,4705.680062
3,AMMA-CATCH::AMMA-CATCH__Nalohou-Mid,9.74530,1.60530,1945,674,2992.014891
4,AMMA-CATCH::AMMA-CATCH__Nalohou-Top,9.74407,1.60580,1945,674,3014.100764
5,AMMA-CATCH::AMMA-CATCH__Tondikiboro,13.54800,2.69560,1956,621,4727.466955
6,AMMA-CATCH::AMMA-CATCH__Wankama,13.64560,2.63200,1956,620,2775.703200
7,ARM::ARM__Anthony,37.21340,-98.09690,877,320,2076.575734
8,ARM::ARM__Ashton,37.13300,-97.26600,886,321,3172.609348
9,ARM::ARM__Byron,36.88100,-98.28500,875,324,3249.884221


count    2210.000000
mean     3409.818882
std      1302.078771
min       103.825157
25%      2486.754698
50%      3490.617712
75%      4461.233783
max      6210.330489
Name: smap_distance_m, dtype: float64

In [22]:
SMAP_STATION_MAP = (
    OUTPUT_DIR
    / "station_smap_grid_mapping.parquet"
)

stations_smap.to_parquet(
    SMAP_STATION_MAP,
    index=False,
)

print("Saved:", SMAP_STATION_MAP)

Saved: Data/Global/smap/station_smap_grid_mapping.parquet


In [23]:
def extract_granule_station_values(
    granule,
    stations_subset,
):
    """
    Extract sm_surface from one SPL4SMGP V008 granule
    at the requested station grid cells.

    Returns:
        np.ndarray of shape (n_stations,)
    """

    remote_files = earthaccess.open(
        [granule],
        show_progress=False,
    )

    if not remote_files:
        raise RuntimeError(
            "earthaccess.open() returned no file."
        )

    f = remote_files[0]

    with xr.open_dataset(
        f,
        engine="h5netcdf",
        group="Geophysical_Data",
        phony_dims="access",
        cache=False,
    ) as geo:

        sm = geo[SMAP_VARIABLE]

        # The final two dimensions are the EASE-Grid Y/X dimensions.
        y_dim = sm.dims[-2]
        x_dim = sm.dims[-1]

        y_idx = xr.DataArray(
            stations_subset[
                "smap_y_idx"
            ].to_numpy(
                dtype=np.int32
            ),
            dims="station_index",
        )

        x_idx = xr.DataArray(
            stations_subset[
                "smap_x_idx"
            ].to_numpy(
                dtype=np.int32
            ),
            dims="station_index",
        )

        values = (
            sm.isel(
                {
                    y_dim: y_idx,
                    x_dim: x_idx,
                }
            )
            .values
            .astype(np.float32)
        )

    values[
        ~np.isfinite(values)
    ] = np.nan

    values[
        (values < SM_MIN)
        |
        (values > SM_MAX)
    ] = np.nan

    return values

In [24]:
test_values = extract_granule_station_values(
    test_granules[0],
    stations_smap,
)

test_result = pd.DataFrame({
    "station_key": stations_smap[
        "station_key"
    ].to_numpy(),

    "SMAP_3h": test_values,
})

print(
    "Shape:",
    test_result.shape,
)

display(
    test_result.head()
)

/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/earthaccess/store.py:516: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum([granule.size() for granule in granules]) / 1024, 2)


Shape: (2210, 2)


,station_key,SMAP_3h
0,AMMA-CATCH::AMMA-CATCH__Banizoumbou,0.029536
1,AMMA-CATCH::AMMA-CATCH__Belefoungou-Mid,0.195613
2,AMMA-CATCH::AMMA-CATCH__Belefoungou-Top,0.198726
3,AMMA-CATCH::AMMA-CATCH__Nalohou-Mid,0.200129
4,AMMA-CATCH::AMMA-CATCH__Nalohou-Top,0.200129


In [25]:
valid_count = (
    test_result["SMAP_3h"]
    .notna()
    .sum()
)

print(
    "Valid:",
    valid_count,
    "/",
    len(test_result),
)

print(
    "Valid percentage:",
    f"{100 * valid_count / len(test_result):.2f}%"
)

display(
    test_result["SMAP_3h"].describe()
)

if valid_count == 0:
    raise RuntimeError(
        "Still getting zero valid SMAP values. "
        "Do not start production extraction."
    )

# The test dataset is no longer needed.
geo_ds.close()

print("SMAP single-granule test passed.")

Valid: 2210 / 2210
Valid percentage: 100.00%


count    2210.000000
mean        0.232079
std         0.123514
min         0.021884
25%         0.138911
50%         0.213832
75%         0.293025
max         0.797541
Name: SMAP_3h, dtype: float64

SMAP single-granule test passed.


In [26]:
"""def get_smap_granules_for_date(date):
    
    Search all SPL4SMGP V008 granules for one UTC calendar day.
    

    date = pd.Timestamp(date).normalize()

    next_day = date + pd.Timedelta(days=1)

    return earthaccess.search_data(
        short_name=SMAP_SHORT_NAME,
        version=SMAP_VERSION,
        temporal=(
            date.strftime("%Y-%m-%dT00:00:00"),
            next_day.strftime("%Y-%m-%dT00:00:00"),
        ),
        count=-1,
    )"""

'def get_smap_granules_for_date(date):\n\n    Search all SPL4SMGP V008 granules for one UTC calendar day.\n\n\n    date = pd.Timestamp(date).normalize()\n\n    next_day = date + pd.Timedelta(days=1)\n\n    return earthaccess.search_data(\n        short_name=SMAP_SHORT_NAME,\n        version=SMAP_VERSION,\n        temporal=(\n            date.strftime("%Y-%m-%dT00:00:00"),\n            next_day.strftime("%Y-%m-%dT00:00:00"),\n        ),\n        count=-1,\n    )'

In [27]:
"""def extract_smap_day(
    date,
    stations_smap,
):
    
    Extract all available 3-hourly SMAP observations for one UTC day,
    then average them to a daily surface-soil-moisture value per station.
    

    date = pd.Timestamp(date).normalize()

    granules = get_smap_granules_for_date(date)

    if not granules:
        return pd.DataFrame({
            "station_key": stations_smap["station_key"],
            "sample_date": date,
            "SMAP": np.nan,
            "SMAP_n_obs": 0,
        })

    parts = []

    for granule in granules:

        try:
            values = extract_granule_station_values(
                granule,
                stations_smap,
            )

            parts.append(
                values["SMAP_3h"].to_numpy()
            )

        except Exception as exc:
            print(
                f"Warning: failed granule "
                f"{granule}: {exc}"
            )

    if not parts:
        return pd.DataFrame({
            "station_key": stations_smap["station_key"],
            "sample_date": date,
            "SMAP": np.nan,
            "SMAP_n_obs": 0,
        })

    values = np.vstack(parts)

    valid = np.isfinite(values)

    count = valid.sum(axis=0)

    values_for_mean = np.where(
        valid,
        values,
        np.nan,
    )

    daily_mean = np.nanmean(
        values_for_mean,
        axis=0,
    )

    result = pd.DataFrame({
        "station_key": stations_smap["station_key"].to_numpy(),
        "sample_date": date,
        "SMAP": daily_mean.astype(np.float32),
        "SMAP_n_obs": count.astype(np.int8),
    })

    return result"""

'def extract_smap_day(\n    date,\n    stations_smap,\n):\n\n    Extract all available 3-hourly SMAP observations for one UTC day,\n    then average them to a daily surface-soil-moisture value per station.\n\n\n    date = pd.Timestamp(date).normalize()\n\n    granules = get_smap_granules_for_date(date)\n\n    if not granules:\n        return pd.DataFrame({\n            "station_key": stations_smap["station_key"],\n            "sample_date": date,\n            "SMAP": np.nan,\n            "SMAP_n_obs": 0,\n        })\n\n    parts = []\n\n    for granule in granules:\n\n        try:\n            values = extract_granule_station_values(\n                granule,\n                stations_smap,\n            )\n\n            parts.append(\n                values["SMAP_3h"].to_numpy()\n            )\n\n        except Exception as exc:\n            print(\n                f"Warning: failed granule "\n                f"{granule}: {exc}"\n            )\n\n    if not parts:\n        return pd.Da

In [28]:
sample_dates = (
    samples["sample_date"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print(
    "Unique ISMN dates:",
    len(sample_dates)
)

print(
    "Range:",
    sample_dates[0],
    "→",
    sample_dates[-1],
)

Unique ISMN dates: 4204
Range: 2015-04-01 00:00:00 → 2026-10-03 00:00:00


In [29]:
date_to_station_keys = {}

for date, group in samples.groupby("sample_date"):
    date_to_station_keys[
        pd.Timestamp(date)
    ] = set(
        group["station_key"]
        .astype(str)
    )

print(
    "Dates:",
    len(date_to_station_keys)
)

Dates: 4204


In [30]:
# ============================================================
# CELL 30 — BULK SMAP GRANULE INDEX + STATION LOOKUP
# ============================================================

print(
    "Performing one bulk CMR search for SMAP granules..."
)

search_start = sample_dates[0].strftime(
    "%Y-%m-%dT00:00:00"
)

search_end = (
    sample_dates[-1]
    + pd.Timedelta(days=1)
).strftime(
    "%Y-%m-%dT00:00:00"
)

# ------------------------------------------------------------
# ONE bulk Earthdata search for the entire ISMN period
# ------------------------------------------------------------

all_granules = earthaccess.search_data(
    short_name=SMAP_SHORT_NAME,
    version=SMAP_VERSION,
    temporal=(
        search_start,
        search_end,
    ),
    count=-1,
)

print(
    "Granules found:",
    f"{len(all_granules):,}"
)

# ------------------------------------------------------------
# Requested ISMN dates
# ------------------------------------------------------------

requested_dates = {
    pd.Timestamp(d).normalize()
    for d in sample_dates
}

# ------------------------------------------------------------
# Group SMAP granules by UTC calendar date
# ------------------------------------------------------------

granules_by_date = {}

for g in all_granules:

    dt_str = (
        g["umm"]
        ["TemporalExtent"]
        ["RangeDateTime"]
        ["BeginningDateTime"]
    )

    dt = (
        pd.to_datetime(
            dt_str,
            utc=True,
        )
        .tz_localize(None)
        .normalize()
    )

    if dt in requested_dates:

        granules_by_date.setdefault(
            dt,
            [],
        ).append(g)

# ------------------------------------------------------------
# Validate date coverage
# ------------------------------------------------------------

dates_with_granules = (
    set(
        granules_by_date.keys()
    )
)

dates_without_granules = (
    requested_dates
    - dates_with_granules
)

print(
    "Sample dates with SMAP granules:",
    f"{len(dates_with_granules):,}"
)

print(
    "Sample dates without SMAP granules:",
    f"{len(dates_without_granules):,}"
)

if dates_without_granules:

    print(
        "First missing dates:",
        sorted(
            dates_without_granules
        )[:10]
    )

    raise RuntimeError(
        "Some ISMN sample dates have no "
        "SPL4SMGP V008 granules."
    )

# ------------------------------------------------------------
# Count granules
# ------------------------------------------------------------

total_indexed_granules = sum(
    len(v)
    for v in granules_by_date.values()
)

print(
    "Indexed SMAP granules:",
    f"{total_indexed_granules:,}"
)

# ------------------------------------------------------------
# Fast station lookup
# ------------------------------------------------------------

stations_smap_lookup = (
    stations_smap
    .set_index(
        "station_key",
        verify_integrity=True,
    )
)

print(
    "Station lookup rows:",
    f"{len(stations_smap_lookup):,}"
)

print(
    "Bulk SMAP setup ready."
)

Performing one bulk CMR search for SMAP granules...


/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


Granules found: 33,634
Sample dates with SMAP granules: 4,204
Sample dates without SMAP granules: 0
Indexed SMAP granules: 33,632
Station lookup rows: 2,210
Bulk SMAP setup ready.


In [31]:
"""def extract_smap_day_for_samples(
    date,
    stations_smap,
    requested_station_keys,
):
   
    Extract daily SMAP only for stations that actually have
    an ISMN sample on this date.
   

    date = pd.Timestamp(date).normalize()

    requested_station_keys = list(
        requested_station_keys
    )

    station_subset = (
        stations_smap[
            stations_smap["station_key"]
            .isin(requested_station_keys)
        ]
        .reset_index(drop=True)
    )

    granules = get_smap_granules_for_date(date)

    if not granules:
        return pd.DataFrame({
            "station_key": requested_station_keys,
            "sample_date": date,
            "SMAP": np.nan,
            "SMAP_n_obs": 0,
        })

    parts = []

    for granule in granules:
        try:
            remote_files = earthaccess.open(
                [granule],
                show_progress=False,
            )

            f = remote_files[0]

            with xr.open_dataset(
                f,
                engine="h5netcdf",
                group="Geophysical_Data",
                phony_dims="access",
            ) as geo:

                sm = geo["sm_surface"]

                x = geo["x"].values
                y = geo["y"].values

                x_dim = next(
                    d for d in sm.dims
                    if sm.sizes[d] == len(x)
                )

                y_dim = next(
                    d for d in sm.dims
                    if sm.sizes[d] == len(y)
                )

                y_idx = xr.DataArray(
                    station_subset["smap_y_idx"].to_numpy(),
                    dims="station_index",
                )

                x_idx = xr.DataArray(
                    station_subset["smap_x_idx"].to_numpy(),
                    dims="station_index",
                )

                values = (
                    sm.isel({
                        y_dim: y_idx,
                        x_dim: x_idx,
                    })
                    .values
                    .astype(np.float32)
                )

                values[~np.isfinite(values)] = np.nan

                values[
                    (values < SM_MIN)
                    | (values > SM_MAX)
                ] = np.nan

                parts.append(values)

        except Exception as exc:
            print(
                f"Warning: failed granule: {exc}"
            )

    if not parts:
        return pd.DataFrame({
            "station_key": requested_station_keys,
            "sample_date": date,
            "SMAP": np.nan,
            "SMAP_n_obs": 0,
        })

    array = np.vstack(parts)

    valid = np.isfinite(array)

    counts = valid.sum(axis=0)

    daily = np.nanmean(
        np.where(
            valid,
            array,
            np.nan,
        ),
        axis=0,
    )

    result = pd.DataFrame({
        "station_key": station_subset["station_key"],
        "sample_date": date,
        "SMAP": daily.astype(np.float32),
        "SMAP_n_obs": counts.astype(np.int8),
    })

    return result"""

'def extract_smap_day_for_samples(\n    date,\n    stations_smap,\n    requested_station_keys,\n):\n\n    Extract daily SMAP only for stations that actually have\n    an ISMN sample on this date.\n\n\n    date = pd.Timestamp(date).normalize()\n\n    requested_station_keys = list(\n        requested_station_keys\n    )\n\n    station_subset = (\n        stations_smap[\n            stations_smap["station_key"]\n            .isin(requested_station_keys)\n        ]\n        .reset_index(drop=True)\n    )\n\n    granules = get_smap_granules_for_date(date)\n\n    if not granules:\n        return pd.DataFrame({\n            "station_key": requested_station_keys,\n            "sample_date": date,\n            "SMAP": np.nan,\n            "SMAP_n_obs": 0,\n        })\n\n    parts = []\n\n    for granule in granules:\n        try:\n            remote_files = earthaccess.open(\n                [granule],\n                show_progress=False,\n            )\n\n            f = remote_files[0]\n\n  

In [32]:
"""SMAP_OUTPUT = (
    OUTPUT_DIR
    / "smap_by_sample.parquet"
)

print(SMAP_OUTPUT)"""

'SMAP_OUTPUT = (\n    OUTPUT_DIR\n    / "smap_by_sample.parquet"\n)\n\nprint(SMAP_OUTPUT)'

In [33]:
# ============================================================
# SAMPLE KEYS FOR SMAP JOIN
# ============================================================

sample_columns = [
    "sample_id",
    "station_key",
    "station_id",
    "network",
    "sample_date",
]

sample_keys = (
    samples[sample_columns]
    .copy()
)

sample_keys["sample_date"] = (
    pd.to_datetime(
        sample_keys["sample_date"]
    )
    .dt.normalize()
)

sample_keys["station_key"] = (
    sample_keys["station_key"]
    .astype(str)
)

# sample_id must uniquely identify each final sample row.
duplicate_sample_ids = (
    sample_keys["sample_id"]
    .duplicated(keep=False)
)

if duplicate_sample_ids.any():

    display(
        sample_keys.loc[
            duplicate_sample_ids
        ].head(20)
    )

    raise ValueError(
        "Duplicate sample_id values found."
    )

print(
    "Sample rows:",
    f"{len(sample_keys):,}"
)

print(
    "Unique station/date pairs:",
    f"{sample_keys[['station_key', 'sample_date']]}"
    ".drop_duplicates().shape[0]:,}"
)

display(
    sample_keys.head()
)

Sample rows: 3,514,643
Unique station/date pairs:                                  station_key sample_date
0        AMMA-CATCH::AMMA-CATCH__Banizoumbou  2015-04-01
1        AMMA-CATCH::AMMA-CATCH__Banizoumbou  2015-04-02
2        AMMA-CATCH::AMMA-CATCH__Banizoumbou  2015-04-03
3        AMMA-CATCH::AMMA-CATCH__Banizoumbou  2015-04-04
4        AMMA-CATCH::AMMA-CATCH__Banizoumbou  2015-04-05
...                                      ...         ...
3514638            iRON::iRON__Spring_Valley  2025-09-21
3514639            iRON::iRON__Spring_Valley  2025-09-22
3514640            iRON::iRON__Spring_Valley  2025-09-23
3514641            iRON::iRON__Spring_Valley  2025-09-24
3514642            iRON::iRON__Spring_Valley  2025-09-25

[3514643 rows x 2 columns].drop_duplicates().shape[0]:,}


,sample_id,station_key,station_id,network,sample_date
0,0,AMMA-CATCH::AMMA-CATCH__Banizoumbou,AMMA-CATCH__Banizoumbou,AMMA-CATCH,2015-04-01
1,1,AMMA-CATCH::AMMA-CATCH__Banizoumbou,AMMA-CATCH__Banizoumbou,AMMA-CATCH,2015-04-02
2,2,AMMA-CATCH::AMMA-CATCH__Banizoumbou,AMMA-CATCH__Banizoumbou,AMMA-CATCH,2015-04-03
3,3,AMMA-CATCH::AMMA-CATCH__Banizoumbou,AMMA-CATCH__Banizoumbou,AMMA-CATCH,2015-04-04
4,4,AMMA-CATCH::AMMA-CATCH__Banizoumbou,AMMA-CATCH__Banizoumbou,AMMA-CATCH,2015-04-05


In [34]:
# ============================================================
# CELL 34 — PARALLEL SPL4SMGP V008 PRODUCTION DAILY EXTRACTOR
# ============================================================

from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed,
)

import h5py


# ------------------------------------------------------------
# Number of concurrent remote granule reads
# ------------------------------------------------------------
#
# Start conservatively with 2.
#
# We previously saw that aggressive parallelism can make
# remote HDF5 reads slower rather than faster.
#
# Increase only after benchmarking.
# ------------------------------------------------------------

SMAP_IO_WORKERS = 2


def _read_one_smap_granule(
    granule,
    station_y,
    station_x,
):
    """
    Read one SPL4SMGP V008 granule and extract the requested
    station pixels.

    The SPL4SMGP V008 sm_surface variable is stored as one
    full (1624, 3856) HDF5 chunk, so the complete array is
    read once per granule and station extraction is then done
    locally with NumPy.
    """

    remote_files = earthaccess.open(
        [granule],
        show_progress=False,
    )

    if not remote_files:
        raise RuntimeError(
            "earthaccess.open() returned no file."
        )

    f = remote_files[0]

    with h5py.File(
        f,
        mode="r",
        driver="fileobj",
    ) as h5:

        sm_h5 = h5[
            f"Geophysical_Data/{SMAP_VARIABLE}"
        ]

        # ----------------------------------------------------
        # Read complete compressed HDF5 chunk
        # ----------------------------------------------------

        full_sm = np.asarray(
            sm_h5[:],
            dtype=np.float32,
        )

    # --------------------------------------------------------
    # Validate expected global SMAP grid
    # --------------------------------------------------------

    if full_sm.shape != (
        SMAP_NROWS,
        SMAP_NCOLS,
    ):

        raise RuntimeError(
            "Unexpected SMAP array shape: "
            f"{full_sm.shape}; "
            f"expected "
            f"({SMAP_NROWS}, {SMAP_NCOLS})"
        )

    # --------------------------------------------------------
    # Replace HDF5 fill values with NaN
    # --------------------------------------------------------

    full_sm[
        full_sm == -9999.0
    ] = np.nan

    # --------------------------------------------------------
    # Apply valid SMAP physical range
    # --------------------------------------------------------

    full_sm[
        (full_sm < SM_MIN)
        |
        (full_sm > SM_MAX)
    ] = np.nan

    # --------------------------------------------------------
    # Extract requested station pixels
    # --------------------------------------------------------

    values = (
        full_sm[
            station_y,
            station_x,
        ]
        .astype(
            np.float32
        )
    )

    del full_sm

    return values


# ------------------------------------------------------------
# Production daily extractor
# ------------------------------------------------------------

def extract_smap_day_for_samples(
    date,
    stations_smap_lookup,
    requested_station_keys,
):
    """
    Extract all available 3-hourly SPL4SMGP V008 observations
    for one UTC calendar day and average them per station.

    The remote granules for the day are processed concurrently
    using SMAP_IO_WORKERS threads.

    Returns:
        DataFrame with:
            station_key
            sample_date
            SMAP
            SMAP_n_obs
    """

    date = pd.Timestamp(
        date
    ).normalize()

    requested_station_keys = sorted(
        map(
            str,
            requested_station_keys,
        )
    )

    # ========================================================
    # Resolve required stations
    # ========================================================

    missing_keys = (
        pd.Index(
            requested_station_keys
        )
        .difference(
            stations_smap_lookup.index
        )
    )

    if len(missing_keys) > 0:

        raise KeyError(
            "Stations missing from stations_smap: "
            + ", ".join(
                map(
                    str,
                    missing_keys[:10],
                )
            )
        )

    station_subset = (
        stations_smap_lookup
        .loc[
            requested_station_keys
        ]
        .reset_index()
    )

    # ========================================================
    # Find granules for this date
    # ========================================================

    granules = granules_by_date.get(
        date,
        [],
    )

    if not granules:

        return pd.DataFrame({

            "station_key":
                station_subset[
                    "station_key"
                ].to_numpy(),

            "sample_date":
                date,

            "SMAP":
                np.nan,

            "SMAP_n_obs":
                np.zeros(
                    len(
                        station_subset
                    ),
                    dtype=np.int8,
                ),
        })

    # ========================================================
    # Build station grid indices once
    # ========================================================

    station_y = (
        station_subset[
            "smap_y_idx"
        ]
        .to_numpy(
            dtype=np.int32
        )
    )

    station_x = (
        station_subset[
            "smap_x_idx"
        ]
        .to_numpy(
            dtype=np.int32
        )
    )

    # ========================================================
    # Submit all granules concurrently
    # ========================================================

    futures = {}

    with ThreadPoolExecutor(
        max_workers=SMAP_IO_WORKERS
    ) as executor:

        for granule_number, granule in enumerate(
            granules
        ):

            future = executor.submit(
                _read_one_smap_granule,
                granule,
                station_y,
                station_x,
            )

            futures[
                future
            ] = granule_number

        # ----------------------------------------------------
        # Collect results
        # ----------------------------------------------------

        ordered_parts = [
            None
        ] * len(granules)

        for future in as_completed(
            futures
        ):

            granule_number = futures[
                future
            ]

            try:

                values = future.result()

                ordered_parts[
                    granule_number
                ] = values

            except Exception as exc:

                raise RuntimeError(
                    f"Failed SMAP granule "
                    f"{granule_number + 1}/"
                    f"{len(granules)} "
                    f"for {date.date()}: "
                    f"{exc}"
                ) from exc

    # ========================================================
    # Require at least one successfully read granule
    # ========================================================

    parts = [
        p
        for p in ordered_parts
        if p is not None
    ]

    if not parts:

        raise RuntimeError(
            f"No SMAP granules were successfully "
            f"processed for {date.date()}"
        )

    # ========================================================
    # Stack 3-hourly observations
    # ========================================================

    array = np.vstack(
        parts
    )

    # ========================================================
    # Count valid observations
    # ========================================================

    valid = np.isfinite(
        array
    )

    counts = (
        valid
        .sum(
            axis=0
        )
        .astype(
            np.int8
        )
    )

    # ========================================================
    # Daily mean
    # ========================================================

    sums = np.nansum(
        array,
        axis=0,
        dtype=np.float32,
    )

    daily = np.full(
        counts.shape,
        np.nan,
        dtype=np.float32,
    )

    has_values = (
        counts > 0
    )

    daily[
        has_values
    ] = (
        sums[
            has_values
        ]
        /
        counts[
            has_values
        ]
    )

    # ========================================================
    # Return daily station-level result
    # ========================================================

    return pd.DataFrame({

        "station_key":
            station_subset[
                "station_key"
            ].to_numpy(),

        "sample_date":
            date,

        "SMAP":
            daily,

        "SMAP_n_obs":
            counts,
    })


print(
    "Parallel SPL4SMGP V008 production extractor ready."
)

print(
    "Concurrent granule workers:",
    SMAP_IO_WORKERS,
)

Parallel SPL4SMGP V008 production extractor ready.
Concurrent granule workers: 2


In [35]:
"""# Replace SMAP_OUTPUT with a directory path
SMAP_DATASET_DIR = "Data/Global/pipeline_stage1/smap_parquet_dataset"
os.makedirs(SMAP_DATASET_DIR, exist_ok=True)"""

'# Replace SMAP_OUTPUT with a directory path\nSMAP_DATASET_DIR = "Data/Global/pipeline_stage1/smap_parquet_dataset"\nos.makedirs(SMAP_DATASET_DIR, exist_ok=True)'

In [ ]:
# ============================================================
# BENCHMARK PARALLEL SPL4SMGP EXTRACTION
# ============================================================

benchmark_date = pd.Timestamp(
    sample_dates[0]
).normalize()

benchmark_stations = date_to_station_keys[
    benchmark_date
]

print(
    "Benchmark date:",
    benchmark_date.date(),
)

print(
    "Stations:",
    len(benchmark_stations),
)

print(
    "Granules:",
    len(
        granules_by_date[
            benchmark_date
        ]
    ),
)

t0 = time.perf_counter()

benchmark_daily = (
    extract_smap_day_for_samples(
        benchmark_date,
        stations_smap_lookup,
        benchmark_stations,
    )
)

elapsed = (
    time.perf_counter()
    - t0
)

print(
    "\nElapsed:",
    f"{elapsed:.2f} s",
)

print(
    "Rows:",
    len(benchmark_daily),
)

print(
    "Valid SMAP:",
    benchmark_daily[
        "SMAP"
    ].notna().sum(),
    "/",
    len(benchmark_daily),
)

print(
    "Mean observations:",
    benchmark_daily[
        "SMAP_n_obs"
    ].mean(),
)

display(
    benchmark_daily.head()
)

Benchmark date: 2015-04-01
Stations: 1166
Granules: 8


/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/earthaccess/store.py:516: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum([granule.size() for granule in granules]) / 1024, 2)


In [ ]:
# ============================================================
# CELL 37 — CHECKPOINTED SMAP PRODUCTION EXTRACTION
# ============================================================

from tqdm.auto import tqdm

errors = []


for date in tqdm(
    sample_dates,
    desc="Extracting SMAP",
):

    date = pd.Timestamp(
        date
    ).normalize()

    date_str = date.strftime(
        "%Y-%m-%d"
    )

    out_file = (
        SMAP_DATASET_DIR
        / f"smap_{date_str}.parquet"
    )

    # --------------------------------------------------------
    # Completed date -> skip
    # --------------------------------------------------------

    if out_file.exists():
        continue

    needed_stations = (
        date_to_station_keys[
            date
        ]
    )

    tmp_file = (
        SMAP_DATASET_DIR
        / f".smap_{date_str}.parquet.tmp"
    )

    try:

        # ----------------------------------------------------
        # Extract one UTC day of SPL4SMGP observations
        # ----------------------------------------------------

        daily = (
            extract_smap_day_for_samples(
                date,
                stations_smap_lookup,
                needed_stations,
            )
        )

        daily = daily[
            [
                "station_key",
                "sample_date",
                "SMAP",
                "SMAP_n_obs",
            ]
        ]

        # ----------------------------------------------------
        # Attach sample_id and metadata
        # ----------------------------------------------------

        daily = sample_keys.merge(
            daily,
            on=[
                "station_key",
                "sample_date",
            ],
            how="inner",
            validate="many_to_one",
        )

        # ----------------------------------------------------
        # IMPORTANT: validate this date immediately
        # ----------------------------------------------------

        expected_rows = (
            sample_keys[
                sample_keys[
                    "sample_date"
                ] == date
            ]
            .shape[0]
        )

        if len(daily) != expected_rows:

            raise RuntimeError(
                f"SMAP row-count mismatch "
                f"for {date.date()}: "
                f"expected {expected_rows}, "
                f"got {len(daily)}"
            )

        # ----------------------------------------------------
        # Final daily schema
        # ----------------------------------------------------

        daily = daily[
            [
                "sample_id",
                "station_key",
                "station_id",
                "network",
                "sample_date",
                "SMAP",
                "SMAP_n_obs",
            ]
        ]

        # ----------------------------------------------------
        # Write temporary file first
        # ----------------------------------------------------

        if tmp_file.exists():
            tmp_file.unlink()

        daily.to_parquet(
            tmp_file,
            index=False,
            compression="zstd",
        )

        # ----------------------------------------------------
        # Atomic checkpoint
        # ----------------------------------------------------

        os.replace(
            tmp_file,
            out_file,
        )

    except Exception as exc:

        if tmp_file.exists():
            tmp_file.unlink()

        errors.append({
            "sample_date": date,
            "error": repr(exc),
        })

        print(
            f"\nFAILED {date.date()}: {exc}"
        )

        # Continue with next date.
        continue


# ============================================================
# Finished
# ============================================================

print(
    "\nExtraction loop finished."
)

if errors:

    error_df = pd.DataFrame(
        errors
    )

    error_path = (
        SMAP_DATASET_DIR
        / "smap_extraction_errors.csv"
    )

    error_df.to_csv(
        error_path,
        index=False,
    )

    print(
        f"Failed dates: "
        f"{len(error_df):,}"
    )

    print(
        "Error log:",
        error_path,
    )

else:

    print(
        "No extraction errors."
    )

Extracting SMAP:   0%|          | 0/4204 [00:00<?, ?it/s]/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/earthaccess/store.py:516: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum([granule.size() for granule in granules]) / 1024, 2)
Extracting SMAP:   0%|          | 0/4204 [00:50<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# ============================================================
# LOAD AND VALIDATE COMPLETED SMAP DATASET
# ============================================================

expected_sample_ids = set(
    sample_keys["sample_id"]
)

smap_final = pd.read_parquet(
    SMAP_DATASET_DIR
)

print(
    "SMAP rows:",
    f"{len(smap_final):,}"
)

print(
    "Unique samples:",
    smap_final["sample_id"].nunique()
)

# Every ISMN sample must appear exactly once.
if (
    len(smap_final)
    != len(sample_keys)
):
    raise RuntimeError(
        f"SMAP row count mismatch: "
        f"expected {len(sample_keys):,}, "
        f"got {len(smap_final):,}"
    )

if (
    smap_final["sample_id"].nunique()
    != len(sample_keys)
):
    raise RuntimeError(
        "SMAP output does not contain "
        "exactly one row per sample_id."
    )

missing_sample_ids = (
    expected_sample_ids
    - set(smap_final["sample_id"])
)

if missing_sample_ids:
    raise RuntimeError(
        f"SMAP output is missing "
        f"{len(missing_sample_ids):,} sample IDs."
    )

display(
    smap_final.head()
)

SMAP rows: 3,514,643
Unique samples: 3514643


,sample_id,station_key,station_id,network,sample_date,SMAP,SMAP_n_obs
0,749012,PBO_H2O::PBO_H2O__PARKINSONF,PBO_H2O__PARKINSONF,PBO_H2O,2015-04-01,NaN,0
1,122313,COSMOS-UK::COSMOS-UK__Bickley_Hall,COSMOS-UK__Bickley_Hall,COSMOS-UK,2015-04-01,NaN,0
2,1267880,SCAN::SCAN__Lovelock_NNR,SCAN__Lovelock_NNR,SCAN,2015-04-01,NaN,0
3,1405838,SCAN::SCAN__Princeton_#1,SCAN__Princeton_#1,SCAN,2015-04-01,NaN,0
4,773640,PBO_H2O::PBO_H2O__Upington,PBO_H2O__Upington,PBO_H2O,2015-04-01,NaN,0


In [ ]:
coverage = {
    "total_rows": len(smap_final),

    "nonmissing_smap": int(
        smap_final["SMAP"].notna().sum()
    ),

    "missing_smap": int(
        smap_final["SMAP"].isna().sum()
    ),

    "missing_percent": float(
        smap_final["SMAP"].isna().mean() * 100
    ),

    "mean_observations_per_day": float(
        smap_final["SMAP_n_obs"].mean()
    ),
}

coverage

{'total_rows': 3514643,
 'nonmissing_smap': 0,
 'missing_smap': 3514643,
 'missing_percent': 100.0,
 'mean_observations_per_day': 0.0}

In [ ]:
print(
    smap_final["SMAP"]
    .describe()
)
invalid = (
    smap_final["SMAP"].notna()
    &
    ~smap_final["SMAP"].between(
        SM_MIN,
        SM_MAX,
    )
)

print(
    "Invalid SMAP values:",
    invalid.sum()
)

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: SMAP, dtype: float64
Invalid SMAP values: 0


In [ ]:
smap_final["SMAP_reprocessing_flag"] = (
    smap_final["sample_date"].between(
        SMAP_REPROCESS_START,
        SMAP_REPROCESS_END,
        inclusive="both",
    )
)

print(
    smap_final[
        "SMAP_reprocessing_flag"
    ].value_counts()
)

SMAP_reprocessing_flag
False    3451396
True       63247
Name: count, dtype: int64


In [ ]:
master_with_smap = samples.merge(
    smap_final[
        [
            "sample_id",
            "SMAP",
            "SMAP_n_obs",
            "SMAP_reprocessing_flag",
        ]
    ],
    on="sample_id",
    how="left",
    validate="one_to_one",
)

print(
    "Master rows:",
    f"{len(master_with_smap):,}"
)

display(
    master_with_smap.head()
)

Master rows: 3,514,643


,sample_id,station_id,sample_date,network,network_name,station,latitude,longitude,elevation_m,climate_zone,source_row_count,min_depth_from_m,max_depth_to_m,sensors,depth_classes,station_key,SMAP,SMAP_n_obs,SMAP_reprocessing_flag
0,0,AMMA-CATCH__Banizoumbou,2015-04-01,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05,0.05,Campbell-CS616,near_surface,AMMA-CATCH::AMMA-CATCH__Banizoumbou,NaN,0,False
1,1,AMMA-CATCH__Banizoumbou,2015-04-02,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05,0.05,Campbell-CS616,near_surface,AMMA-CATCH::AMMA-CATCH__Banizoumbou,NaN,0,False
2,2,AMMA-CATCH__Banizoumbou,2015-04-03,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05,0.05,Campbell-CS616,near_surface,AMMA-CATCH::AMMA-CATCH__Banizoumbou,NaN,0,False
3,3,AMMA-CATCH__Banizoumbou,2015-04-04,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05,0.05,Campbell-CS616,near_surface,AMMA-CATCH::AMMA-CATCH__Banizoumbou,NaN,0,False
4,4,AMMA-CATCH__Banizoumbou,2015-04-05,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05,0.05,Campbell-CS616,near_surface,AMMA-CATCH::AMMA-CATCH__Banizoumbou,NaN,0,False


: 

In [ ]:
MASTER_SMAP_OUTPUT = (
    OUTPUT_DIR
    / "global_ismn_smap_index.parquet"
)

master_with_smap.to_parquet(
    MASTER_SMAP_OUTPUT,
    index=False,
)

print(
    "Saved:",
    MASTER_SMAP_OUTPUT
)